对应 `tf.keras` 的01~02章节

In [25]:
import torch

x = torch.arange(4.0)
x
x.requires_grad_(True)  # 等价于x=torch.arange(4.0,requires_grad=True)
x.grad  # 默认值是None
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [26]:
import matplotlib as mpl
import matplotlib.pyplot as plt
#是一个 Jupyter Notebook 的魔术命令，用于在 Notebook 中内嵌显示 matplotlib 图形。当你在代码单元格中运行这个命令后，所有使用 matplotlib 绘制的图形都会直接显示在 Notebook 中，而不是弹出单独的窗口。
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 2.2.6
pandas 2.3.3
sklearn 1.8.0
torch 2.9.1+cpu
cpu


In [ ]:
28*28#计算用，只是看看结果而已

## 数据准备

In [ ]:
import numpy as np
X = np.arange(24).reshape(2, 3, 4)#构造三维
X

按照特定轴进行算法

In [ ]:
import numpy as np
X = np.arange(12).reshape(3, 4)#构造二维
print(X)
b = np.sum(X,axis=0)#0轴代表是一列一列的
c = np.sum(X,axis=1)#1轴代表是一行一行看
print("-"*50)
print(b)
print("-"*50)
print(c)

In [ ]:
x = np.arange(4)
# x, x.sum()
x.shape,x.sum()
print(type(x.shape))#输出x.shape的类型，是一个元组
# 一维数组
a = np.array([1, 2, 3])
print(a.shape)  # (3,)

# 二维数组（3行2列）
b = np.array([[1, 2], [3, 4], [5, 6]])
print(b.shape)  # (3, 2)

# 三维数组
c = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print(c.shape)  # (2, 2, 2)

非降维求和

In [ ]:
import numpy as np
A = np.arange(20).reshape(5, 4)
print(f'A为{A}')
sum_A = A.sum(axis=1, keepdims=True)#轴1为行，keepdims这个参数表示保持数组的维度。
print(sum_A)
sum_A = A.sum(axis=1, keepdims=False)
sum_A

In [ ]:
# 张量是描述具有任意数量轴的维数组的通用方法在深度学习中，​​
# 一切数据皆为张量​​。你之前学的 NumPy 数组操作，在张量上几乎完全通用。
# 矩阵转置
import numpy as np
a = np.arange(9).reshape(3,3)#这个是确定性
a = np.random.randint(0,10,(3,3))#这是随机
print(a)
print('-'*50)
print(a.T)
print('-'*50)
print(a==a.T)

if np.array_equal(a,a.T):#为啥不能用a==a.T因为返回的是数组bool型
  print(f'为转置矩阵')
else:
  print(f'不是转置矩阵')


In [ ]:
a = np.random.randint(0,10,(3,3))#这是随机
b = np.random.randint(0,10,(3,3))#这是随机
print(a)
print('-'*50)
print(b)
c = a.T+b.T
print('-'*50)
print(c)
d = (a+b).T
print('-'*50)
print(d)
print('-'*50)
print(c==d)


In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision import transforms


# 定义数据集的变换
transform = transforms.Compose([
    transforms.ToTensor(), # 转换为tensor，进行归一化
    # transforms.Normalize(mean, std) # 标准化，mean和std是数据集的均值和方差
])
# fashion_mnist图像分类数据集，衣服分类，60000张训练图片，10000张测试图片
train_ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_ds = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

# torchvision 数据集里没有提供训练集和验证集的划分
# 当然也可以用 torch.utils.data.Dataset 实现人为划分

In [ ]:
type(train_ds)

In [ ]:
len(train_ds)

In [ ]:
type(train_ds[0]) #元组

In [ ]:
# 通过id取数据，取到的是一个元祖,是第一个样本,在训练时，把特征和标签分开
img, label = train_ds[0]
# print(img)
print(label)
img.shape
# img.shape = (1, 28, 28)，这是因为通道数在最前面，灰度图片只有1个通道：亮度/灰度值

In [ ]:
type(img) #tensor中文是 张量,和numpy的ndarray类似

In [ ]:
img[0]

In [ ]:
img

In [ ]:
#计算均值和方差
def cal_mean_std(ds):
    mean = 0.
    std = 0.
    for img, _ in ds: # 遍历每张图片,img.shape=[1,28,28] img, _ 中的 _ 表示忽略标签
        mean += img.mean(dim=(1, 2))#dim=(1, 2) 表示沿着高度和宽度两个维度计算
        std += img.std(dim=(1, 2))
    mean /= len(ds)  # 除以图片总数
    std /= len(ds)   # 除以图片总数
    return mean, std
# 假设 img = [[[1, 2],     # 形状：[1, 2, 2] 的简单例子
#         [3, 4]]]

# 计算 mean(dim=(1, 2)):
# 1. 展平像素：1, 2, 3, 4
# 2. 计算均值：(1+2+3+4)/4 = 2.5
# 3. 结果：tensor([2.5])


print(cal_mean_std(train_ds))


In [ ]:
type(img)

In [ ]:
label

In [ ]:
type(img) #tensor中文是 张量,和numpy的ndarray类似

In [ ]:
label

In [ ]:
# 显示图片，这里需要把transforms.ToTensor(),进行归一化注释掉，否则是不行的
def show_img_content(img):
    from PIL import Image

    # 打开一个图像文件
    # img = Image.open(img)


    print("图像大小:", img.size)
    print("图像模式:", img.mode)


    # 如果图像是单通道的，比如灰度图，你可以这样获取像素值列表：
    if img.mode == 'L':
        pixel_values = list(img.getdata())
        print(pixel_values)
show_img_content(img) #这里必须把上面的 transforms.ToTensor(), # 转换为tensor，进行归一化注释掉，否则是不行的

In [ ]:
#这个代码必须是注释了上面的 transforms.ToTensor()才能够运行的
def show_single_image(img_arr):
    plt.imshow(img_arr, cmap="binary") # 显示图片
    plt.colorbar() # 显示颜色条
    plt.show()


show_single_image(img)

In [ ]:
def show_imgs(n_rows, n_cols, train_ds, class_names):
    assert n_rows * n_cols < len(train_ds)  #确保打印的图片小于总样本数
    plt.figure(figsize = (n_cols * 1.4, n_rows * 1.6))  #宽1.4高1.6，宽，高
    for row in range(n_rows):
        for col in range(n_cols):
            index = n_cols * row + col  # 计算索引，从0开始
            plt.subplot(n_rows, n_cols, index+1)#因为从1开始
            img_arr, label = train_ds[index]
            img_arr = np.transpose(img_arr, (1, 2, 0))  # 通道换到最后一维
            plt.imshow(img_arr, cmap="binary",
                       interpolation = 'nearest')#interpolation='nearest'是临近插值
            plt.axis('off')#去除坐标系
            plt.title(class_names[label]) # 显示类别名称
    plt.show()



#已知的图片类别
# lables在这个路径https://github.com/zalandoresearch/fashion-mnist
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress',
               'Coat', 'Sandal', 'Shirt', 'Sneaker',
               'Bag', 'Ankle boot'] #0-9分别代表的类别
#只是打印了前15个样本
show_imgs(3, 5, train_ds, class_names)


In [ ]:
# 从数据集到dataloader
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True) #batch_size分批，shuffle洗牌
val_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

在PyTorch中，`DataLoader`是一个迭代器，它封装了数据的加载和预处理过程，使得在训练机器学习模型时可以方便地批量加载数据。`DataLoader`主要负责以下几个方面：

1. **批量加载数据**：`DataLoader`可以将数据集（Dataset）切分为更小的批次（batch），每次迭代提供一小批量数据，而不是单个数据点。这有助于模型学习数据中的统计依赖性，并且可以更高效地利用GPU等硬件的并行计算能力。

2. **数据打乱**：默认情况下，`DataLoader`会在每个epoch（训练周期）开始时打乱数据的顺序。这有助于模型训练时避免陷入局部最优解，并且可以提高模型的泛化能力。

3. **多线程数据加载**：`DataLoader`支持多线程（通过参数`num_workers`）来并行地加载数据，这可以显著减少训练过程中的等待时间，尤其是在处理大规模数据集时。

4. **数据预处理**：`DataLoader`可以与`transforms`结合使用，对加载的数据进行预处理，如归一化、标准化、数据增强等操作。

5. **内存管理**：`DataLoader`负责管理数据的内存使用，确保在训练过程中不会耗尽内存资源。

6. **易用性**：`DataLoader`提供了一个简单的接口，可以很容易地集成到训练循环中。



## 定义模型

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() # 继承父类的初始化方法，子类有父类的属性
        self.flatten = nn.Flatten()  # 展平层
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(784, 300),  # in_features=784, out_features=300, 784是输入特征数，300是输出特征数
            nn.ReLU(), # 激活函数
            nn.Linear(300, 100),#隐藏层神经元数100
            nn.ReLU(), # 激活函数
            nn.Linear(100, 10),#输出层神经元数10
        )

    def forward(self, x): # 前向计算
        # x.shape [batch size, 1, 28, 28]
        x = self.flatten(x)
        # 展平后 x.shape [batch size, 784]
        logits = self.linear_relu_stack(x)
        # logits.shape [batch size, 10]
        return logits #没有经过softmax,称为logits

model = NeuralNetwork()

In [ ]:
# 看看网络结构
model
# Linear层参数 = (输入维度 × 输出维度) + 输出维度
          #  = 输出维度 × (输入维度 + 1)  [如果有偏置]

In [ ]:
784*300+300+300*100+100+100*10+10

In [ ]:
for name, param in model.named_parameters(): # 打印模型参数
      print(f'{name}----{param.shape}')

In [ ]:
# 看看模型参数
list(model.parameters())  # 这种方法拿到模型的所有可学习参数,requires_grad=True


In [ ]:
# model.state_dict()  # 这种方法用于保存模型参数，看能看见参数属于模型的哪一部分

## 训练

pytorch的训练需要自行实现，包括
1. 定义损失函数
2. 定义优化器
3. 定义训练步
4. 训练

In [ ]:
# 1. 定义损失函数 采用交叉熵损失
loss_fct = nn.CrossEntropyLoss() #内部先做softmax，然后计算交叉熵
# 2. 定义优化器 采用SGD
# Optimizers specified in the torch.optim package,随机梯度下降
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from sklearn.metrics import accuracy_score

@torch.no_grad() # 装饰器，禁止反向传播，节省内存
def evaluating(model, dataloader, loss_fct):
    loss_list = [] # 记录损失
    pred_list = [] # 记录预测
    label_list = [] # 记录标签
    for datas, labels in dataloader:#10000/32=312
        datas = datas.to(device) # 转到GPU
        labels = labels.to(device) # 转到GPU
        # 前向计算
        logits = model(datas)
        loss = loss_fct(logits, labels)         # 验证集损失
        loss_list.append(loss.item()) # 记录损失

        preds = logits.argmax(axis=-1)    # 验证集预测,argmax返回最大值索引 axis=-1 表示在最后一个维度上操作（类别维度）
        # print(preds)
        pred_list.extend(preds.cpu().numpy().tolist())#将PyTorch张量转换为NumPy数组。只有当张量在CPU上时，这个转换才是合法的
        # print(preds.cpu().numpy().tolist())
        label_list.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(label_list, pred_list) # 计算准确率，这里用的内置函数，当然可以自己实现
    return np.mean(loss_list), acc


In [ ]:
1875*20

In [ ]:
# 训练
def training(model, train_loader, val_loader, epoch, loss_fct, optimizer, eval_step=500):
    record_dict = {
        "train": [],
        "val": []
    }

    global_step = 0
    model.train()
    with tqdm(total=epoch * len(train_loader)) as pbar: # 进度条 1875*20,60000/32=1875
        for epoch_id in range(epoch): # 训练epoch次
            # training
            for datas, labels in train_loader: #执行次数是60000/32=1875
                datas = datas.to(device) #datas尺寸是[batch_size,1,28,28]
                labels = labels.to(device) #labels尺寸是[batch_size]
                # 梯度清空 因为pytorch中梯度是自动累加的
                optimizer.zero_grad()
                # 模型前向计算
                logits = model(datas)
                # 计算损失
                loss = loss_fct(logits, labels)
                # 梯度回传，loss.backward()会计算梯度，loss对模型参数求导
                loss.backward()
                # 调整优化器，包括学习率的变动等,优化器的学习率会随着训练的进行而减小，更新w,b
                optimizer.step() #梯度是计算并存储在模型参数的 .grad 属性中，优化器使用这些存储的梯度来更新模型参数

                preds = logits.argmax(axis=-1) # 训练集预测
                acc = accuracy_score(labels.cpu().numpy(), preds.cpu().numpy())   # 计算准确率，numpy可以
                loss = loss.cpu().item() # 损失转到CPU，item()取值,一个数值
                # record

                record_dict["train"].append({
                    "loss": loss, "acc": acc, "step": global_step
                }) # 记录训练集信息，每一步的损失，准确率，步数

                # evaluating
                if global_step % eval_step == 0:
                    model.eval() # 进入评估模式
                    val_loss, val_acc = evaluating(model, val_loader, loss_fct)
                    record_dict["val"].append({
                        "loss": val_loss, "acc": val_acc, "step": global_step
                    })
                    model.train() # 进入训练模式

                # udate step
                global_step += 1 # 全局步数加1
                pbar.update(1) # 更新进度条
                pbar.set_postfix({"epoch": epoch_id}) # 设置进度条显示信息

    return record_dict


epoch = 40 #改为40
model = model.to(device)
record = training(model, train_loader, val_loader, epoch, loss_fct, optimizer, eval_step=1000)

In [ ]:
record["train"][-5:]

In [ ]:
record["val"][-5:]

In [ ]:
#画线要注意的是损失是不一定在零到1之间的
def plot_learning_curves(record_dict, sample_step=1000):
    # build DataFrame
    train_df = pd.DataFrame(record_dict["train"]).set_index("step").iloc[::sample_step]
    val_df = pd.DataFrame(record_dict["val"]).set_index("step")
    last_step = train_df.index[-1] # 最后一步的步数
    # print(train_df.columns)
    print(train_df['acc'])
    print(val_df['acc'])
    # plot
    fig_num = len(train_df.columns) # 画几张图,分别是损失和准确率
    fig, axs = plt.subplots(1, fig_num, figsize=(5 * fig_num, 5))
    for idx, item in enumerate(train_df.columns):
        # print(train_df[item].values)
        axs[idx].plot(train_df.index, train_df[item], label=f"train_{item}")
        axs[idx].plot(val_df.index, val_df[item], label=f"val_{item}")
        axs[idx].grid() # 显示网格
        axs[idx].legend() # 显示图例
        axs[idx].set_xticks(range(0, train_df.index[-1], 5000)) # 设置x轴刻度
        axs[idx].set_xticklabels(map(lambda x: f"{int(x/1000)}k", range(0, last_step, 5000))) # 设置x轴标签
        axs[idx].set_xlabel("step")

    plt.show()

plot_learning_curves(record)  #横坐标是 steps

## 评估

In [ ]:
# dataload for evaluating

model.eval() # 进入评估模式
loss, acc = evaluating(model, val_loader, loss_fct)
print(f"loss:     {loss:.4f}\naccuracy: {acc:.4f}")